# 09 · Evaluation: metrics for every run × evaluation set × language (× generator family)
**Purpose:** convert saved per-document scores into one long, tidy metrics table plus seed-aggregated values with bootstrap 95% CIs. No model is loaded here.

**Evaluation-set composition** (human side, machine side):
- clean: `test` = (test humans, test machines); `val` likewise
- attacks on machine text only: `T1_para`, `T2_bt`, `T3_h05`, `T3_h15`, `T4_t05`, `T4_t15` = (clean test humans, attacked machines)
- attacks on both classes: `T3_h05both`, … = (attacked humans, attacked machines)
- defence: `*_def` and `test_def` (Unicode normalisation before scoring)
- external: `ext_coling_raw`, `ext_coling_norm`

**Thresholds** (for F1/FPR/ASR) are chosen on *validation* data only: `src` = validation data of the training languages (realistic cross-lingual deployment); `tgt` = validation data of the evaluated language (in-language calibration upper bound). Zero-shot detectors have no training language → `src` = pooled validation data of all languages.

**Outputs:** `08_results/metrics_long.csv`, `08_results/metrics_agg.csv`.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "09_evaluate"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
B_CI = 1000                     # bootstrap resamples for descriptive CIs here (hypothesis tests in 10 use config B)
EVAL = load_eval_texts()
reg = load_registry(); reg = reg[reg.status == "done"].copy()
spec_langs = {(e, s["name"]): s["train_langs"] for e, specs in CFG["experiments"].items() for s in specs}
log.info(f"done runs: {len(reg)}")

def eval_defs(available):
    d = {"val": ("val", "val"), "test": ("test", "test"), "test_def": ("test_def", "test_def"),
         "ext_coling_raw": ("ext_coling_raw", "ext_coling_raw"), "ext_coling_norm": ("ext_coling_norm", "ext_coling_norm"),
         "T1_para": ("test", "T1_para"), "T2_bt": ("test", "T2_bt")}
    for tag in ("T3_h05", "T3_h15", "T4_t05", "T4_t15"):
        d[tag] = ("test", tag); d[tag + "both"] = (tag + "both", tag)
    for tag in ("T3_h05", "T3_h15"):
        d[tag + "_def"] = ("test_def", tag + "_def"); d[tag + "both_def"] = (tag + "both_def", tag + "_def")
    return {k: v for k, v in d.items() if v[0] in available and v[1] in available}

DEFS = eval_defs(set(EVAL.eval_set))
META = EVAL.set_index("doc_id")

def assemble(scores, name):
    """Return frame (doc_id, orig_doc_id, lang, label, family, score, prob) for an evaluation set name."""
    hs, ms = DEFS[name]
    h = EVAL[(EVAL.eval_set == hs) & (EVAL.label == 0)]; m = EVAL[(EVAL.eval_set == ms) & (EVAL.label == 1)]
    d = pd.concat([h, m])[["doc_id", "orig_doc_id", "lang", "label", "family"]]
    return d.merge(scores, on="doc_id", how="inner")

## Per-run metrics

In [ ]:
rows = []
for r in reg.itertuples():
    sc = pd.read_parquet(pred_path(r.run_id))
    langs_train = spec_langs.get((r.exp, r.train), LANGS)
    val = assemble(sc, "val")
    val_src = val[val.lang.isin(langs_train)]
    thr = {"src": {"f1": best_f1_threshold(val_src.label.values, val_src.score.values),
                   "fpr1": threshold_at_fpr(val_src.label.values, val_src.score.values, 0.01)}}
    for l, g in val.groupby("lang"):
        thr[l] = {"f1": best_f1_threshold(g.label.values, g.score.values), "fpr1": threshold_at_fpr(g.label.values, g.score.values, 0.01)}
    clean = assemble(sc, "test").set_index("orig_doc_id").score
    for name in DEFS:
        d = assemble(sc, name)
        if d.empty or name == "val":
            continue
        groups = [("all", "all", d)] + [(l, "all", g) for l, g in d.groupby("lang")]
        if name in ("test", "ext_coling_raw", "ext_coling_norm"):
            for l, g in d.groupby("lang"):
                for fam in sorted(set(g.family) - {"human"}):
                    groups.append((l, fam, g[(g.label == 0) | (g.family == fam)]))
        for lang, fam, g in groups:
            y, s = g.label.values, g.score.values
            if y.sum() == 0 or (1 - y).sum() == 0:
                continue
            tt = thr.get(lang, thr["src"]) if lang != "all" else thr["src"]
            row = {"run_id": r.run_id, "exp": r.exp, "detector": r.detector, "train": r.train, "heldout": r.heldout,
                   "seed": r.seed, "eval": name, "lang": lang, "family_subset": fam, "n_pos": int(y.sum()), "n_neg": int((1 - y).sum()),
                   "auroc": auroc(y, s), "tpr_at_1fpr_oracle": tpr_at_fpr(y, s, 0.01), "tpr_at_5fpr_oracle": tpr_at_fpr(y, s, 0.05),
                   "f1_src": macro_f1(y, s, thr["src"]["f1"]), "f1_tgt": macro_f1(y, s, tt["f1"]),
                   "fpr_at_src1": float((s[y == 0] > thr["src"]["fpr1"]).mean()), "tpr_at_src1": float((s[y == 1] > thr["src"]["fpr1"]).mean()),
                   "ece": ece(y, g.prob.values) if g.prob.notna().all() else np.nan}
            if name not in ("test", "val", "test_def") and not name.startswith("ext"):
                mm = g[g.label == 1]
                base = clean.reindex(mm.orig_doc_id).values
                ok = ~np.isnan(base)
                row["asr_src"] = attack_success_rate(base[ok], mm.score.values[ok], thr["src"]["f1"])
            rows.append(row)
long = pd.DataFrame(rows)
long.to_csv(P["results"] / "metrics_long.csv", index=False)
log.info(f"metrics_long rows: {len(long)}")

## Seed aggregation with hierarchical bootstrap CIs (resample documents, average over seeds)

In [ ]:
agg = []
keys = ["exp", "detector", "train", "heldout"]
for k, runs in reg.groupby(keys):
    S = {r.run_id: pd.read_parquet(pred_path(r.run_id)).set_index("doc_id") for r in runs.itertuples()}
    for name in DEFS:
        if name == "val":
            continue
        base = assemble(next(iter(S.values())).reset_index(), name)
        if base.empty:
            continue
        subsets = [(l, g) for l, g in base.groupby("lang")] + [("all", base)]
        for lang, g in subsets:
            y = g.label.values
            if y.sum() == 0 or (1 - y).sum() == 0:
                continue
            M = np.vstack([s.reindex(g.doc_id).score.values for s in S.values()])
            if np.isnan(M).any():
                continue
            a, lo, hi = boot_ci_auroc(y, M, B=B_CI, seed=0)
            agg.append(dict(zip(keys, k)) | {"eval": name, "lang": lang, "n_seeds": len(S), "auroc": a, "ci_lo": lo, "ci_hi": hi,
                                             "auroc_seed_sd": float(np.std([auroc(y, m) for m in M], ddof=1)) if len(S) > 1 else 0.0})
agg = pd.DataFrame(agg); agg.to_csv(P["results"] / "metrics_agg.csv", index=False)
log.info(f"metrics_agg rows: {len(agg)}")
agg[(agg["eval"] == "test") & (agg.lang != "all")].pivot_table(index=["exp", "detector", "train", "heldout"], columns="lang", values="auroc").round(3)